## Structured Output:

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing.

#### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions and nested strucutes.

In [3]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")  # reasoning model

In [4]:
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000015C3C0527B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000015C3C052E40>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title : str = Field(description= "The title of the movie")
    year : int = Field(description= "this year the movie was released")
    director : str = Field(description="The directory of the movie")
    rating : float = Field(description="The movie's rating out of 10")

In [7]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000015C3C0527B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000015C3C052E40>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'this year the movie was released', 'type': 'integer'}, 'director': {'description': 'The directory of the movie', 'type': 'string'}, 'rating': {'description': "The movie's rating out of 10", 'type': 'number'}}, 'required': ['title', 'year',

In [8]:
model.invoke("provide details about the movie Inception")

AIMessage(content="<think>\nOkay, let's see. The user wants details about the movie Inception. First, I need to recall some key points about the film. Directed by Christopher Nolan, right? Starring Leonardo DiCaprio, who plays Dom Cobb. The movie is about dreams and reality. There's some concept where they enter people's dreams to steal or plant ideas.\n\nI should start with the basic info: title, director, release year, cast. Then maybe a plot summary. The main character, Cobb, is a thief who steals secrets through dream-sharing. The team plans to plant an idea in a target's subconscious. The structure of the movie is a bit complex, with layers of dreams within dreams. There's a time dilation element where each level of the dream is slower, so a few minutes in the real world could be hours or days in the dream.\n\nThe cast includes famous actors like Joseph Gordon-Levitt, Ellen Page, and Tom Hardy. Each has significant roles. The visual effects are a big part of the movie, especially 

In [9]:
model_with_structure.invoke("provide details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [10]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    '''A movie with details'''
    title : str = Field(description= "The title of the movie")
    year : int = Field(description= "this year the movie was released")
    director : str = Field(description="The directory of the movie")
    rating : float = Field(description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw= True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check what tools I have available. There\'s a Movie function that requires title, year, director, and rating. I need to make sure I have all that information for Inception.\n\nFirst, the title is obviously "Inception". The year it was released was 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yes, IMDb does have it at 8.8. So I have all the required parameters. I\'ll structure the tool call with these details. Make sure the JSON is correctly formatted with the right data types: year as integer, rating as a number. No missing fields, since all are required. Alright, that should cover it.\n', 'tool_calls': [{'id': 'csn1v27d3', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response

### Nested Structure

In [11]:
class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title:str
    year : int
    cast : list[Actor]
    genres : list[str]
    budget : float | None = Field(None, description= "Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide detials about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito')], genres=['Science Fiction', 'Action', 'Thriller'], budget=160.0)